In [ ]:
import pandas as pd

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

In [ ]:
from ml.features.resample_usage import build_bucketed_series

In [ ]:
fact = pd.read_parquet("../data/interim/fact_pod_events.parquet")
fact.head()

In [ ]:
# Dense window found during exploration: creation events are near-silent
# before this, then dense through the end of the trace.
WINDOW_START = 9891350
WINDOW_END = 12901761

candidate_seconds = {
    "1min": 60,
    "5min": 300,
    "10min": 600,
    "15min": 900,
    "30min": 1800,
    "60min": 3600,
}

In [ ]:
print(f"{'bucket':<8} {'n_buckets':>10} {'pct_zero':>9} {'mean':>10} {'std':>10} {'cv':>6} {'lag1_autocorr':>14}")
for label, secs in candidate_seconds.items():
    series = build_bucketed_series(fact, "cpu_milli", WINDOW_START, WINDOW_END, secs)
    usage = series["cpu_milli"]
    pct_zero = (usage == 0).mean() * 100
    mean = usage.mean()
    std = usage.std()
    cv = std/mean if mean > 0 else float("nan")
    autocorr = usage.autocorr(lag=1)
    print(f"{label:<8} {len(usage):>10} {pct_zero:>8.1f}% {mean:>10.0f} {std:>10.0f} {cv:>6.2f} {autocorr:>14.3f}")

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
series = build_bucketed_series(fact, "cpu_milli", WINDOW_START, WINDOW_END, 3600)  # try 1-hour first
series.head()

In [ ]:
series["day"] = (series["bucket_start"] - WINDOW_START) / 86400
series.head()

In [ ]:
series[series["day"].isin([0, 1, 2, 3, 4, 5])]

In [ ]:
series.plot(x="day", y="cpu_milli", title="CPU demand over the dense window")
plt.xlabel("Day into dense window")
plt.ylabel("cpu_milli in use")
plt.show()